In [4]:
# llamaindex
# !pip install llama-index-core

In [3]:
# !pip install llama-index-llms-ollama


In [6]:
# !pip install llama-index-embeddings-ollama

In [7]:
from llama_index.llms.ollama import Ollama # 라마인덱스 라이브러리에서 Ollama LLM모델 불러오기
from llama_index.embeddings.ollama import OllamaEmbedding # 라마인덱스 Ollama 임베딩 모델 불러오기
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader # 라마인덱스의 벡터스토어 인덱스와 디렉토리 리더 불러오기 

In [ ]:
# Ollama 모델 사용 설정 
llm = Ollama(
    model = "gemma2:2b", # 작은 모델 
    request_timeout=120, # 요청 타임아웃 설정
    temperature=0.5, # 생성된 텍스트의 다양성 조절_ 0.5는 중간 정도의 다양성, 낮은 값은 더 일관된 텍스트, 높은 값은 더 창의적인 텍스트 생성
)
embed_model = OllamaEmbedding(
    model_name="nomic-embed-text", # Ollama에서 제공하는 텍스트 임베딩 모델 사용_ 아까 터미널에서 설치 해서 쓸 수 있는 거 
    )

In [11]:
# 문서 불러오기
documents = SimpleDirectoryReader(input_dir = "../Data/pdf_sample1").load_data() # 문서가 저장된 디렉토리에서 문서 불러오기

In [14]:
documents

[Document(id_='686125fc-eba8-4def-a240-a9ef915ccd35', embedding=None, metadata={'file_path': '/Users/dusik/Documents/WorkSpace/RAG/Note/../Data/pdf_sample1/240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf', 'file_name': '240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf', 'file_type': 'application/pdf', 'file_size': 359946, 'creation_date': '2026-05-18', 'last_modified_date': '2026-05-18'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='%PDF-1.4\n%\n10 0 obj\n<< /Type /Page\n/Parent 1 0 R\n/MediaBox [ 0 0 532 745 ]\n/TrimBox [ 0 0 532 745 ]\n/BleedBox [ 0 0 532 745 ]\n/Resources 9 0 R\n/Contents 16 0 R\n>>\nendobj\n16

In [21]:
%pip install llama-index-readers-file pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 15.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [llama-index-readers-file]ndex-readers-file]
Note: you may need to restart the kernel to use updated packages.


In [23]:
# 문서 불러오기 
documents = SimpleDirectoryReader(input_dir='../Data/pdf_sample1').load_data()

In [24]:
print(documents[1])

Doc ID: 8efbc4bc-03c9-400a-9929-b726f00d9ff2
Text: THE AI REPORT  2024-3 | 2024. 9.11.    「The AI Report」는 인공지능
기술‧산업‧정책의 글로벌 이슈와 동향, 시사점을 적시에 분석, 인공지능 현안에 빠르게 대응하고 관련 정책을 지원하기 위해
한국지능정보사회진흥원(NIA)에서 기획‧발간하고 있습니다.1.본 보고서는 방송통신발전기금으로 수행하는 정보통신·방송 연구개발
사업의 결과물이므로, 보고서 내용을 발표할 때는 반드시 과학기술정보통신부 정보통신·방송 연구개발 사업의 연구 결과임을 밝혀야
합니다.2.한국지능정보사회진흥원(NIA)의 승인 없이 본 보고서의 무단전재를 금하며, 가공·인용할 때는 반드시 출처를
「한국지능정보사회진흥원...


In [25]:
# 문서로 부터 벡터 스토어 인덱스 생성
index = VectorStoreIndex.from_documents(documents, llm=llm, embed_model=embed_model)

2026-05-18 14:22:46,362 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 14:22:47,021 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 14:22:47,663 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 14:22:48,678 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 14:22:48,720 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


In [26]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm) # llm을 쿼리 엔진에 연결하여 질문에 대한 답변 생성_ LLM에 연결 해줘야 질문에 대한 답변을 생성할 수 있음

2026-05-18 14:23:27,089 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [28]:
# 응답 생성
response = query_engine.query("미국의 인공지능 정책과 주요 변화에 대해 알려주세요") # 쿼리 엔진에 질문을 던져서 답변 생성
print(response)
# 위에 temperature=0으로 주면 답변이 더 일관되고 정확하게 나올 수 있음

2026-05-18 14:25:30,421 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-18 14:25:44,570 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


미국은 인공지능 (AI) 관련 정책을 강력하게 추진하고 있습니다.  2023년부터 AI 안전성 확보를 위한 행정명령(Executive Order on the Safe, Secure, and Trustworthy Development & Use of AI)이 발표되어 있으며, 2024년에 AI 활용에 관한 정부 기준 준수 및 책임 있는 대처를 요구하는 규칙 제정안을 통해 미국의 AI 개발 환경을 조성하고 있습니다.  

AI 관련 법률안은 이미 80개 이상 제출되었으며, 의회에서 AI 규제 방안에 대해 논의가 이루어져 있습니다. 공화당 의원들은 AI 규제가 혁신을 저해할 수 있다고 우려하며, 민주당 의원들은 과도한 AI 규제가 미국의 경쟁력을 훼손할 것이라고 주장하고 있습니다.  

미국은 AI 활용에 대한 정책과 관련하여 다양한 정책들을 제안하고 있으며, 이는 국방부의 AI 기술 활용 및 안전성 확보를 위한 노력을 포함합니다. 또한 미국에서는 AI 규제 방식을 검토하고 AI 위험 해결을 위한 권고 사항을 제공하는 초당파 AI 전문가 위원회 설립, AI 훈련 프로그램 확립, CAIO(Chief AI Officer) 임명 의무화, 그리고 인공지능 생성 콘텐츠에 대한 사용자 인식 표시를 의무화 등의 정책들을 제안하고 있습니다. 






In [29]:
# metadata 확인
response.metadata

{'f46e172d-e6b3-44d4-9fd3-c7e7db93036a': {'page_label': '18',
  'file_name': '240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf',
  'file_path': '/Users/dusik/Documents/WorkSpace/RAG/Note/../Data/pdf_sample1/240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf',
  'file_type': 'application/pdf',
  'file_size': 359946,
  'creation_date': '2026-05-18',
  'last_modified_date': '2026-05-18'},
 '269cfe20-551f-4267-9557-ad5b8560fced': {'page_label': '5',
  'file_name': '240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf',
  'file_path': '/Users/dusik/Documents/WorkSpace/RAG/Note/../Data/pdf_sample1/240828_(AI리포트)_미국의_인공지능(AI)_정책,전략.pdf',
  'file_type': 'application/pdf',
  'file_size': 359946,
  'creation_date': '2026-05-18',
  'last_modified_date': '2026-05-18'}}

In [30]:
# 정확도
for node in response.source_nodes:
    print(f"score : {node.score}")
    print("-" * 50)

score : 0.6377422773458289
--------------------------------------------------
score : 0.6318372351944067
--------------------------------------------------
